# AMPS I · Week 01 — Lab
## Poisson's equation, numerical quadrature, and your first DOLFINx solver

| | |
|---|---|
| **Lecture sections** | Part B (Quadrature) · Part C (Poisson in DOLFINx) · Part D (LLM workflow) |
| **Time** | ≈ 80 min in class |
| **You will** | run FEniCSx in Colab under MPI · verify quadrature exactness with Basix · solve a mixed-BC Poisson problem · measure convergence rates |

**How this notebook works.** Cells that start with `%%fenicsx` run DOLFINx in a *separate MPI process*;
variables do **not** survive between them. Computation cells save results to `/tmp/fnx/`,
and ordinary cells load and plot them with matplotlib.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## 1 · Hello, MPI
`-np 4` launches four processes. Each one owns part of the mesh.

In [ ]:
%%fenicsx -np 4
from mpi4py import MPI
import dolfinx
from dolfinx import mesh

comm = MPI.COMM_WORLD
msh = mesh.create_unit_square(comm, 64, 64)
n_local = msh.topology.index_map(2).size_local          # cells owned by this rank
n_total = comm.allreduce(n_local)
print(f"rank {comm.rank}/{comm.size}: owns {n_local} of {n_total} cells", flush=True)
if comm.rank == 0:
    print("dolfinx", dolfinx.__version__)

## 2 · Numerical quadrature with Basix  *(lecture Part B)*
### 2.1 Gauss rules: $m$ points are exact up to degree $2m-1$

In [ ]:
%%fenicsx -np 1
import basix
import numpy as np

print(" m  pts  err(x^(2m-1))  err(x^(2m))")
for m in range(1, 6):
    pts, wts = basix.make_quadrature(basix.CellType.interval, 2 * m - 1)
    x = pts[:, 0]
    e1 = abs(wts @ x**(2 * m - 1) - 1 / (2 * m))
    e2 = abs(wts @ x**(2 * m) - 1 / (2 * m + 1))
    print(f"{m:2d}  {len(wts):3d}  {e1:13.1e}  {e2:11.1e}")

### 2.2 Triangle rules: check every monomial $x^a y^b$, $a+b \le k$
Exact value on the reference triangle: $\displaystyle\int x^a y^b = \frac{a!\,b!}{(a+b+2)!}$.

In [ ]:
%%fenicsx -np 1
import basix
import numpy as np
from math import factorial

def max_monomial_error(k):
    pts, wts = basix.make_quadrature(basix.CellType.triangle, k)
    err = 0.0
    for a in range(k + 1):
        for b in range(k + 1 - a):
            exact = factorial(a) * factorial(b) / factorial(a + b + 2)
            err = max(err, abs(wts @ (pts[:, 0]**a * pts[:, 1]**b) - exact))
    return len(wts), err

for k in (1, 2, 4, 6, 8):
    n, err = max_monomial_error(k)
    print(f"degree {k}: {n:2d} points, max error {err:.1e}")

### 2.3 Why not equally spaced points? Newton–Cotes weights
Closed Newton–Cotes weights solve the Vandermonde system $\sum_q w_q x_q^j = \frac{1}{j+1}$.
This is plain NumPy, so it runs in an ordinary cell.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3.5))
for n in (3, 5, 7, 9, 11):
    xs = np.linspace(0, 1, n)
    w = np.linalg.solve(np.vander(xs, increasing=True).T, 1 / np.arange(1, n + 1))
    ax.plot(xs, w, "o-", label=f"{n} pts (min w = {w.min():.3f})")
ax.axhline(0, color="k", lw=0.8); ax.legend(fontsize=8); ax.set_title("Closed Newton–Cotes weights")
plt.show()

### 2.4 Quadrature inside DOLFINx
DOLFINx picks a degree automatically; you can override it with `metadata={"quadrature_degree": q}`.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
import ufl
from dolfinx import fem, mesh

comm = MPI.COMM_WORLD
msh = mesh.create_unit_square(comm, 32, 32)
x = ufl.SpatialCoordinate(msh)
exact = (2 / np.pi)**2
qs, errs = [], []
for q in range(1, 11):
    dxq = ufl.Measure("dx", domain=msh, metadata={"quadrature_degree": q})
    val = comm.allreduce(fem.assemble_scalar(fem.form(
        ufl.sin(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1]) * dxq)), op=MPI.SUM)
    qs.append(q); errs.append(abs(val - exact))
if comm.rank == 0:
    np.savez("/tmp/fnx/w01_qdeg.npz", q=qs, err=errs)
    print("saved /tmp/fnx/w01_qdeg.npz")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

d = np.load("/tmp/fnx/w01_qdeg.npz")
plt.figure(figsize=(6, 3.5))
plt.semilogy(d["q"], d["err"], "o-")
plt.xlabel("quadrature degree"); plt.ylabel("|error|")
plt.title(r"$\int_\Omega \sin\pi x \sin\pi y$ on a 32×32 mesh"); plt.grid(alpha=0.3)
plt.show()

> **✏️ Your turn (5 min).** Why does the error stop improving around $10^{-15}$? Why are degrees 2 and 3 almost the same? *(Hint: look at the triangle point counts printed in 2.2.)*

## 3 · From weak form to DOLFINx  *(lecture Part C)*
Model problem (lecture §A1): $-\Delta u = f$ on $(0,1)^2$, $u=0$ on $x=0,1$, $\partial u/\partial n = 0$ on $y=0,1$,
with $u_{ex} = \sin(\pi x)\cos(\pi y)$ and $f = 2\pi^2 u_{ex}$.

| Math | DOLFINx 0.11 |
|---|---|
| triangulation $\mathscr{T}_h$ | `mesh.create_unit_square(comm, N, N)` |
| $V_h$ = continuous P1 | `fem.functionspace(msh, ("Lagrange", 1))` |
| $a(u,v) = \int \nabla u\cdot\nabla v$ | `ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx` |
| $u_h = 0$ on $x=0,1$ (essential) | `fem.dirichletbc(...)` |
| $\partial u/\partial n = 0$ (natural) | *nothing* |
| $K\mathbf{u} = \mathbf{f}$ | `LinearProblem(...).solve()` |

In [ ]:
%%fenicsx -np 1
from mpi4py import MPI
import numpy as np
import ufl
from dolfinx import fem, mesh, default_scalar_type
from dolfinx.fem.petsc import LinearProblem

comm = MPI.COMM_WORLD

def solve_poisson(N, degree=1):
    """Mixed-BC Poisson on the unit square.
    Weak form: find u_h in V_h (u_h = 0 on x=0,1) such that
        (∇u_h, ∇v) = (f, v)  for all v in V_h (v = 0 on x=0,1).
    Neumann data ∂u/∂n = 0 on y=0,1 is natural (no boundary term)."""
    msh = mesh.create_unit_square(comm, N, N)
    V = fem.functionspace(msh, ("Lagrange", degree))
    x = ufl.SpatialCoordinate(msh)
    u_ex = ufl.sin(ufl.pi * x[0]) * ufl.cos(ufl.pi * x[1])
    f = 2 * ufl.pi**2 * u_ex                       # f = -Δu_ex
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    a = ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx
    L = f * v * ufl.dx
    fdim = msh.topology.dim - 1
    walls = mesh.locate_entities_boundary(
        msh, fdim, lambda x: np.isclose(x[0], 0.0) | np.isclose(x[0], 1.0))
    bc = fem.dirichletbc(default_scalar_type(0.0),
                         fem.locate_dofs_topological(V, fdim, walls), V)
    problem = LinearProblem(a, L, bcs=[bc], petsc_options_prefix=f"poisson_{N}_{degree}_",
                            petsc_options={"ksp_type": "preonly", "pc_type": "lu"})
    uh = problem.solve()
    e = uh - u_ex
    L2 = np.sqrt(comm.allreduce(fem.assemble_scalar(fem.form(e**2 * ufl.dx)), op=MPI.SUM))
    H1 = np.sqrt(comm.allreduce(fem.assemble_scalar(
        fem.form(ufl.inner(ufl.grad(e), ufl.grad(e)) * ufl.dx)), op=MPI.SUM))
    return msh, V, uh, L2, H1

msh, V, uh, L2, H1 = solve_poisson(16)
print(f"16×16 P1: ||u-u_h||_L2 = {L2:.3e},  |u-u_h|_H1 = {H1:.3e}")
# save for plotting (single process → the whole mesh is on rank 0)
np.savez("/tmp/fnx/w01_poisson.npz", xy=V.tabulate_dof_coordinates()[:, :2],
         cells=V.dofmap.list, u=uh.x.array)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri

d = np.load("/tmp/fnx/w01_poisson.npz")
tri = mtri.Triangulation(d["xy"][:, 0], d["xy"][:, 1], d["cells"])
fig = plt.figure(figsize=(11, 4))
ax = fig.add_subplot(1, 2, 1); c = ax.tripcolor(tri, d["u"], shading="gouraud", cmap="coolwarm")
ax.triplot(tri, "k-", lw=0.2); ax.set_aspect("equal"); fig.colorbar(c, ax=ax); ax.set_title("$u_h$")
ax = fig.add_subplot(1, 2, 2, projection="3d"); ax.plot_trisurf(tri, d["u"], cmap="coolwarm")
plt.show()

> **✏️ Your turn (10 min).** Copy the cell above and solve with **P2** elements on an **8×8** mesh.
> Compare the $L^2$ error with the 16×16 P1 result. Which costs fewer unknowns for the same accuracy?

In [ ]:
%%fenicsx -np 1
# TODO: solve with degree=2 on an 8×8 mesh and print the errors and the number of DOFs

## 4 · Convergence study under MPI
Theory (Week 7): $\|u-u_h\|_{L^2} = O(h^{k+1})$ and $|u-u_h|_{H^1} = O(h^{k})$ for P$k$.
Observed rate between meshes $h_1, h_2$: $\;r = \log(e_1/e_2)/\log(h_1/h_2)$.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
import ufl
from dolfinx import fem, mesh, default_scalar_type
from dolfinx.fem.petsc import LinearProblem

comm = MPI.COMM_WORLD

def solve_poisson(N, degree=1):
    """Mixed-BC Poisson on the unit square.
    Weak form: find u_h in V_h (u_h = 0 on x=0,1) such that
        (∇u_h, ∇v) = (f, v)  for all v in V_h (v = 0 on x=0,1).
    Neumann data ∂u/∂n = 0 on y=0,1 is natural (no boundary term)."""
    msh = mesh.create_unit_square(comm, N, N)
    V = fem.functionspace(msh, ("Lagrange", degree))
    x = ufl.SpatialCoordinate(msh)
    u_ex = ufl.sin(ufl.pi * x[0]) * ufl.cos(ufl.pi * x[1])
    f = 2 * ufl.pi**2 * u_ex                       # f = -Δu_ex
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    a = ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx
    L = f * v * ufl.dx
    fdim = msh.topology.dim - 1
    walls = mesh.locate_entities_boundary(
        msh, fdim, lambda x: np.isclose(x[0], 0.0) | np.isclose(x[0], 1.0))
    bc = fem.dirichletbc(default_scalar_type(0.0),
                         fem.locate_dofs_topological(V, fdim, walls), V)
    problem = LinearProblem(a, L, bcs=[bc], petsc_options_prefix=f"poisson_{N}_{degree}_",
                            petsc_options={"ksp_type": "preonly", "pc_type": "lu"})
    uh = problem.solve()
    e = uh - u_ex
    L2 = np.sqrt(comm.allreduce(fem.assemble_scalar(fem.form(e**2 * ufl.dx)), op=MPI.SUM))
    H1 = np.sqrt(comm.allreduce(fem.assemble_scalar(
        fem.form(ufl.inner(ufl.grad(e), ufl.grad(e)) * ufl.dx)), op=MPI.SUM))
    return msh, V, uh, L2, H1

Ns = [4, 8, 16, 32, 64]
out = {}
for deg in (1, 2):
    errs = [solve_poisson(N, deg)[3:] for N in Ns]
    out[f"P{deg}"] = np.array(errs)
if comm.rank == 0:
    np.savez("/tmp/fnx/w01_conv.npz", h=1 / np.array(Ns), **out)
    print("saved /tmp/fnx/w01_conv.npz")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

d = np.load("/tmp/fnx/w01_conv.npz"); h = d["h"]
fig, ax = plt.subplots(figsize=(6, 4))
for key, mk in (("P1", "o"), ("P2", "s")):
    e = d[key]
    rL2 = np.log(e[1:, 0] / e[:-1, 0]) / np.log(h[1:] / h[:-1])
    rH1 = np.log(e[1:, 1] / e[:-1, 1]) / np.log(h[1:] / h[:-1])
    print(f"{key}: L2 rates {np.round(rL2, 2)},  H1 rates {np.round(rH1, 2)}")
    ax.loglog(h, e[:, 0], mk + "-", label=f"{key} $L^2$"); ax.loglog(h, e[:, 1], mk + "--", label=f"{key} $H^1$")
ax.set_xticks(h, [f"1/{round(1 / hi)}" for hi in h]); ax.set_xticks([], minor=True)
ax.set_xlabel("h"); ax.set_ylabel("error"); ax.legend(); ax.grid(True, which="both", alpha=0.3)
plt.show()

## 5 · LLM-assisted coding: live demo  *(lecture Part D)*
1. Open your LLM assistant. Paste the **DOLFINx 0.11 API reference sheet** first.
2. Use this prompt (fill in the weak form yourself — that is the point):

```text
Context: [API reference sheet]
Problem: -Δu = f on the unit square, u = 0 on x=0 and x=1, ∂u/∂n = 0 on y=0 and y=1.
Weak form (derived by me): find u in V0 such that ∫∇u·∇v dx = ∫ f v dx for all v in V0.
Spaces: continuous P1 Lagrange.
Task: DOLFINx 0.11 script for %%fenicsx -np 2 that solves the problem with
u_ex = sin(πx)cos(πy), computes the L2 and H1 errors for N = 8, 16, 32, 64,
and prints the observed rates. MPI-safe.
```
3. Paste the generated code below **unchanged**, run it, and then go through the checklist.

In [ ]:
%%fenicsx -np 2
# TODO: paste the code your LLM produced (unchanged) and run it

**Verification checklist** — tick each one before you trust the output:

- [ ] Runs with DOLFINx 0.11 without edits (if not: which API was outdated?)
- [ ] The weak form in the code matches *your* weak form, term by term (signs!)
- [ ] Dirichlet BC applied on **both** $x=0$ and $x=1$; nothing added on $y=0,1$
- [ ] Errors use `comm.allreduce` → same numbers with `-np 1` and `-np 2`
- [ ] Observed rates ≈ 2 ($L^2$) and ≈ 1 ($H^1$)

This is exactly what **HW0 Part B** asks you to do on a deliberately broken script.